VGG16 transfer learning adding new people

In [33]:
from keras.applications import VGG16
from keras.models import Model
from keras.layers import Dense, Dropout, GlobalAveragePooling2D
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import RMSprop
from keras.callbacks import ModelCheckpoint, EarlyStopping
import os

# Load the VGG16 model
rows = 224
cols = 224
model = VGG16(weights='imagenet', include_top=False, input_shape=(rows, cols, 3))

# Freeze the layers of the model
for layer in model.layers:
    layer.trainable = False

# Define the function to add new layers on top of the VGG16 model
def addlayer(bottom_model, num_classes):
    x = bottom_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)
    return x

# Check the number of classes in the training directory
train_data_dir = '/Users/vladpavlovich/Desktop/TrainingTransfer'
validation_data_dir = '/Users/vladpavlovich/Desktop/ValidationTransfer'

num_classes = len(next(os.walk(train_data_dir))[1])  # Count the number of subdirectories (classes)

# Add new layers on top of the VGG16 model
FC_Head = addlayer(model, num_classes)
modelnew = Model(inputs=model.input, outputs=FC_Head)

# Print the summary of the new model
print(modelnew.summary())

# Prepare data generators
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=20,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode='nearest')

validation_datagen = ImageDataGenerator(rescale=1./255)

train_batchsize = 15
val_batchsize = 10

train_generator = train_datagen.flow_from_directory(train_data_dir,
                                                    target_size=(rows, cols),
                                                    batch_size=train_batchsize,
                                                    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(validation_data_dir,
                                                              target_size=(rows, cols),
                                                              batch_size=val_batchsize,
                                                              class_mode='categorical',
                                                              shuffle=False)

# Set up the model checkpoint and early stopping callbacks
checkpoint = ModelCheckpoint("face_recog_vgg.h5", monitor="val_loss", mode="min", save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=3, verbose=1, restore_best_weights=True)
callbacks = [earlystop, checkpoint]

# Compile the model
modelnew.compile(loss='categorical_crossentropy', optimizer=RMSprop(lr=0.001), metrics=['accuracy'])

# Train the model
nb_train_samples = sum([len(files) for r, d, files in os.walk(train_data_dir)])
nb_validation_samples = sum([len(files) for r, d, files in os.walk(validation_data_dir)])
epochs = 15
batch_size = 16

history = modelnew.fit(train_generator,
                       steps_per_epoch=nb_train_samples // batch_size,
                       epochs=epochs,
                       callbacks=callbacks,
                       validation_data=validation_generator,
                       validation_steps=nb_validation_samples // batch_size)

# Save the model
modelnew.save("face_recog_vgg.h5")


Model: "model_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_10 (InputLayer)       [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0   

Epoch 1/15


KeyboardInterrupt: 

With Optimzers

In [ ]:
from keras.applications import VGG16
from keras.models import Model
from keras.layers import Dense, Dropout, GlobalAveragePooling2D
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers.legacy import Adam  # Use legacy Adam optimizer
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os

# Load the VGG16 model
rows = 224
cols = 224
model = VGG16(weights='imagenet', include_top=False, input_shape=(rows, cols, 3))

# Freeze the initial layers of the model
for layer in model.layers[:15]:
    layer.trainable = False

# Unfreeze the top layers of the model
for layer in model.layers[15:]:
    layer.trainable = True

# Define the function to add new layers on top of the VGG16 model
def addlayer(bottom_model, num_classes):
    x = bottom_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)
    return x

# Check the number of classes in the training directory
train_data_dir = '/Users/vladpavlovich/Desktop/TrainingTransfer'
validation_data_dir = '/Users/vladpavlovich/Desktop/ValidationTransfer'

num_classes = len(next(os.walk(train_data_dir))[1])  # Count the number of subdirectories (classes)

# Add new layers on top of the VGG16 model
FC_Head = addlayer(model, num_classes)
modelnew = Model(inputs=model.input, outputs=FC_Head)

# Print the summary of the new model
print(modelnew.summary())

# Prepare data generators
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=40,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode='nearest')

validation_datagen = ImageDataGenerator(rescale=1./255)

train_batchsize = 32
val_batchsize = 16

train_generator = train_datagen.flow_from_directory(train_data_dir,
                                                    target_size=(rows, cols),
                                                    batch_size=train_batchsize,
                                                    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(validation_data_dir,
                                                              target_size=(rows, cols),
                                                              batch_size=val_batchsize,
                                                              class_mode='categorical',
                                                              shuffle=False)

# Set up the model checkpoint, early stopping, and learning rate reduction callbacks
checkpoint = ModelCheckpoint("face_recog_vgg4.h5", monitor="val_loss", mode="min", save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=5, verbose=1, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1, min_lr=0.00001)
callbacks = [earlystop, checkpoint, reduce_lr]

# Compile the model
modelnew.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.0001), metrics=['accuracy'])

# Train the model
nb_train_samples = sum([len(files) for r, d, files in os.walk(train_data_dir)])
nb_validation_samples = sum([len(files) for r, d, files in os.walk(validation_data_dir)])
epochs = 30
batch_size = 32

history = modelnew.fit(train_generator,
                       steps_per_epoch=nb_train_samples // batch_size,
                       epochs=epochs,
                       callbacks=callbacks,
                       validation_data=validation_generator,
                       validation_steps=nb_validation_samples // batch_size)

# Save the model
modelnew.save("face_recog_vgg4.h5")


Model: "model_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_9 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0   

In [ ]:
from keras.models import load_model
classifier = load_model('face_recog_vgg4.h5')

In [37]:
from keras.models import load_model
import os
import cv2
import numpy as np

# Load the trained model
classifier = load_model('face_recog_vgg4.h5')

# Dynamically generate the class labels from the training directory, excluding hidden files
train_data_dir = '/Users/vladpavlovich/Desktop/TrainingTransfer'
class_labels = {str(i): folder for i, folder in enumerate(sorted(folder for folder in os.listdir(train_data_dir) if not folder.startswith('.')))}

def draw_test(name, pred, im):
    actor = class_labels[str(pred[0])]  # Convert prediction to string and access the first element
    BLACK = [0, 0, 0]
    expanded_image = cv2.copyMakeBorder(im, 80, 0, 0, 100, cv2.BORDER_CONSTANT, value=BLACK)
    cv2.putText(expanded_image, actor, (0, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.imshow(name, expanded_image)

def predict_image(image_path):
    # Read and process the image
    input_im = cv2.imread(image_path)
    input_original = input_im.copy()
    input_original = cv2.resize(input_original, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)

    input_im = cv2.resize(input_im, (224, 224), interpolation=cv2.INTER_LINEAR)  # Resize to 224x224
    input_im = input_im / 255.0
    input_im = input_im.reshape(1, 224, 224, 3)

    # Get Prediction
    res = np.argmax(classifier.predict(input_im, verbose=0), axis=1)

    # Print predicted class to console
    predicted_class = class_labels[str(res[0])]
    print(f'Predicted Class: {predicted_class}')

    # Show image with predicted class
    draw_test("Prediction", res, input_original)
    cv2.waitKey(5000)
    cv2.destroyAllWindows()

# Example usage
image_path = '/Users/vladpavlovich/Desktop/MultipleInputsCropped/Brad Pitt/Brad Pitt_15.jpg'  # Replace with the path to your image
predict_image(image_path)


Predicted Class: Brad Pitt


KeyboardInterrupt: 

In [39]:
from keras.models import load_model
import os
import cv2
import numpy as np

# Load the trained model
classifier = load_model('face_recog_vgg4.h5')

# Dynamically generate the class labels from the training directory, excluding hidden files
train_data_dir = '/Users/vladpavlovich/Desktop/TrainingTransfer'
class_labels = {str(i): folder for i, folder in enumerate(sorted(folder for folder in os.listdir(train_data_dir) if not folder.startswith('.')))}

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) == 0:
        return None, None
    (x, y, w, h) = faces[0]
    return image[y:y+w, x:x+h], faces[0]

def draw_test(name, pred, im):
    actor = class_labels[str(pred[0])]  # Convert prediction to string and access the first element
    BLACK = [0, 0, 0]
    expanded_image = cv2.copyMakeBorder(im, 80, 0, 0, 100, cv2.BORDER_CONSTANT, value=BLACK)
    cv2.putText(expanded_image, actor, (0, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.imshow(name, expanded_image)

def predict_image(image_path):
    # Read and process the image
    input_im = cv2.imread(image_path)
    face, rect = detect_face(input_im)
    
    if face is not None:
        input_original = face.copy()
        input_original = cv2.resize(input_original, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)
    
        input_im = cv2.resize(face, (224, 224), interpolation=cv2.INTER_LINEAR)  # Resize to 224x224
        input_im = input_im / 255.0
        input_im = input_im.reshape(1, 224, 224, 3)

        # Get Prediction
        res = np.argmax(classifier.predict(input_im, verbose=0), axis=1)

        # Print predicted class to console
        predicted_class = class_labels[str(res[0])]
        print(f'Predicted Class: {predicted_class}')

        # Show image with predicted class
        draw_test("Prediction", res, input_original)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()
    else:
        print("No face detected in the image.")

# Example usage
image_path = '/Users/vladpavlovich/Desktop/FaceImages/Original Images/Original Images/Brad Pitt/Brad Pitt_3.jpg'  # Replace with the path to your image
predict_image(image_path)


Predicted Class: Brad Pitt


In [40]:
from keras.models import load_model
import os
import cv2
import numpy as np

# Load the trained model
classifier = load_model('face_recog_vgg4.h5')

# Dynamically generate the class labels from the training directory, excluding hidden files
train_data_dir = '/Users/vladpavlovich/Desktop/TrainingTransfer'
class_labels = {str(i): folder for i, folder in enumerate(sorted(folder for folder in os.listdir(train_data_dir) if not folder.startswith('.')))}

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) == 0:
        return None, None
    (x, y, w, h) = faces[0]
    return image[y:y+h, x:x+w], faces[0]

def draw_test(name, pred, im, confidence):
    actor = pred if confidence >= 0.6 else "Unknown"  # Set threshold to 0.6, adjust as needed
    BLACK = [0, 0, 0]
    expanded_image = cv2.copyMakeBorder(im, 80, 0, 0, 100, cv2.BORDER_CONSTANT, value=BLACK)
    cv2.putText(expanded_image, actor, (0, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.imshow(name, expanded_image)

def predict_image(image_path):
    # Read and process the image
    input_im = cv2.imread(image_path)
    face, rect = detect_face(input_im)
    
    if face is not None:
        input_original = face.copy()
        input_original = cv2.resize(input_original, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)
    
        input_im = cv2.resize(face, (224, 224), interpolation=cv2.INTER_LINEAR)  # Resize to 224x224
        input_im = input_im / 255.0
        input_im = input_im.reshape(1, 224, 224, 3)

        # Get Prediction
        predictions = classifier.predict(input_im, verbose=0)
        confidence = np.max(predictions)  # Get the highest confidence score
        res = np.argmax(predictions, axis=1)

        # Print predicted class to console
        predicted_class = class_labels[str(res[0])]
        print(f'Predicted Class: {predicted_class} with confidence {confidence}')

        # Show image with predicted class
        draw_test("Prediction", predicted_class, input_original, confidence)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()
    else:
        print("No face detected in the image.")

# Example usage
image_path = '/Users/vladpavlovich/Desktop/FaceImages/Original Images/Original Images/Brad Pitt/Brad Pitt_3.jpg'  # Replace with the path to your image
predict_image(image_path)


Predicted Class: Brad Pitt with confidence 0.9960219264030457


In [46]:
import tensorflow as tf

tf.config.list_physical_devices('GPU')

[]

In [47]:
from keras.models import load_model
import os
import cv2
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the trained model
classifier = load_model('face_recog_vgg4.h5')

# Dynamically generate the class labels from the training directory, excluding hidden files
train_data_dir = '/Users/vladpavlovich/Desktop/TrainingTransfer'
class_labels = {str(i): folder for i, folder in enumerate(sorted(folder for folder in os.listdir(train_data_dir) if not folder.startswith('.')))}

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) == 0:
        return None, None
    (x, y, w, h) = faces[0]
    return image[y:y+h, x:x+w], faces[0]

def predict_image(image_path):
    # Read and process the image
    input_im = cv2.imread(image_path)
    face, rect = detect_face(input_im)
    
    if face is not None:
        input_im = cv2.resize(face, (224, 224), interpolation=cv2.INTER_LINEAR)  # Resize to 224x224
        input_im = input_im / 255.0
        input_im = input_im.reshape(1, 224, 224, 3)

        # Get Prediction
        predictions = classifier.predict(input_im, verbose=0)
        confidence = np.max(predictions)  # Get the highest confidence score
        res = np.argmax(predictions, axis=1)

        # Print predicted class to console
        predicted_class = class_labels[str(res[0])]
        return predicted_class, confidence
    else:
        return "No face detected", 0

def evaluate_directory(directory_path, target_class="Brad Pitt"):
    y_true = []
    y_pred = []
    confidence_scores = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".jpg") or filename.endswith(".jpeg") or filename.endswith(".png"):
            image_path = os.path.join(directory_path, filename)
            predicted_class, confidence = predict_image(image_path)
            if predicted_class != "No face detected":
                y_true.append(target_class)
                y_pred.append(predicted_class)
                confidence_scores.append(confidence)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=target_class, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=target_class, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=target_class, zero_division=0)
    
    return accuracy, precision, recall, f1, y_true, y_pred, confidence_scores

# Example usage
directory_path = '/Users/vladpavlovich/Desktop/BradPitt' 
accuracy, precision, recall, f1, y_true, y_pred, confidence_scores = evaluate_directory(directory_path)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

# Optionally, print individual predictions and confidences
for true, pred, confidence in zip(y_true, y_pred, confidence_scores):
    print(f'True: {true}, Predicted: {pred}, Confidence: {confidence}')


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].